# 03 · Before/After Change Detection → Baskets
**Owner: P2 (IVP)** · Smart Shelf Market Basket Analysis

Turns shelf photos into transactions - the bridge between vision and AI.

- **Method A - YOLO-based:** detect & count products before and after; removed = before − after
- **Method B - Classical:** colour differencing + morphology on the rectified shelf + planogram (which product sits in which cell)
- **Experiment 3:** precision / recall / F1 of both methods against ground truth
- Output: `results/detected_baskets.csv` → used by notebook 04

**Real photo sessions** go in `DATA_DIR/sessions/` (see README). Until then, demo sessions are generated.

In [ ]:
# ============================================================
# SETUP - identical in every notebook. Works on Colab, Mac, Windows.
# ============================================================
import os, sys, subprocess
from pathlib import Path

# Colab only: where to get the code from.
#   - GitHub (recommended): set with  python tools/set_repo_url.py <url>
#   - "" : the product-identification-correlation folder is in Google Drive at MyDrive/product-identification-correlation
REPO_URL = "https://github.com/behlah-rgb/product-identification-correlation.git"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    if REPO_URL:
        repo = Path("/content/product-identification-correlation")
        url = REPO_URL
        try:  # private repo: add a GitHub token as Colab secret GITHUB_TOKEN
            from google.colab import userdata
            url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
        except Exception:
            pass
        if repo.exists():
            subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
        else:
            subprocess.run(["git", "clone", "-q", url, str(repo)], check=True)
    else:
        repo = Path("/content/drive/MyDrive/product-identification-correlation")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)
else:
    repo = Path.cwd()
    while not (repo / "src").exists() and repo != repo.parent:
        repo = repo.parent

sys.path.insert(0, str(repo))
os.chdir(repo)

from src import env
env.summary()

In [ ]:
import json, cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import change_detect as cd, preprocess as pp, demo_data, detection
from src.env import SESSIONS_DIR, DATA_DIR, RESULTS_DIR, get_device
from src.viz import show, draw_dets

real = (SESSIONS_DIR / "sessions.csv").exists()
root = SESSIONS_DIR if real else DATA_DIR / "sessions_demo"
if not real and not (root / "sessions.csv").exists():
    demo_data.make_demo_sessions(root)
print("Sessions:", "REAL" if real else "DEMO", "->", root)

planogram = json.load(open(root / "planogram.json"))
sessions = pd.read_csv(root / "sessions.csv", dtype=str).fillna("")
sessions["truth"] = sessions["removed"].apply(lambda s: sorted(x.strip() for x in s.split(";") if x.strip()))
print("Planogram:", planogram)
sessions.head()

## 1. Look at one session

In [ ]:
sid = sessions.session_id.iloc[0]
before = cv2.imread(str(root / sid / "before.jpg"))
after = cv2.imread(str(root / sid / "after.jpg"))
b_rect, ok_b = cd.prepare(before)
a_rect, ok_a = cd.prepare(after)
print("Rectified via ArUco:", ok_b, ok_a, "| ground truth removed:", sessions.truth.iloc[0])
show([before, after, pp.draw_grid(b_rect), pp.draw_grid(a_rect)],
     ["before (camera)", "after (camera)", "before (rectified + preprocessed)", "after (rectified + preprocessed)"], cols=4, size=4)

## 2. Method B - classical: colour differencing + morphology
1. Gaussian blur → CIELAB colour difference (lightness down-weighted, so lighting changes between shots are ignored)
2. Threshold ΔE > 30 → morphological opening/closing → binary change mask
3. Per cell: fraction of changed pixels > threshold ⇒ product removed; planogram gives its name

In [ ]:
CHANGE_THRESHOLD = 0.15

res = cd.removed_classical(before, after, planogram, change_threshold=CHANGE_THRESHOLD)
grid = np.zeros((3, 3))
for (r, c), s in res["cell_scores"].items():
    grid[r, c] = s["changed_frac"]

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(res["change_mask"], cmap="gray"); ax[0].set_title("Change mask"); ax[0].axis("off")
im = ax[1].imshow(grid, cmap="Reds", vmin=0, vmax=0.6)
for (r, c), s in res["cell_scores"].items():
    ax[1].text(c, r, f"{s['changed_frac']:.2f}\nSSIM {s['ssim']:.2f}\n{planogram.get(f'{r},{c}', '')[:12]}",
               ha="center", va="center", fontsize=8)
ax[1].set_title(f"Changed-pixel fraction per cell (removed > {CHANGE_THRESHOLD})"); ax[1].axis("off")
plt.colorbar(im, ax=ax[1]); plt.tight_layout(); plt.show()
print("Predicted removed:", res["removed"], "| truth:", sessions.truth.iloc[0])

In [ ]:
pairs = [(cv2.imread(str(root / s / "before.jpg")), cv2.imread(str(root / s / "after.jpg"))) for s in sessions.session_id]

def run_classical(th):
    preds = [cd.removed_classical(b, a, planogram, change_threshold=th)["removed"] for b, a in pairs]
    return preds, cd.aggregate([cd.score_removals(p, t) for p, t in zip(preds, sessions.truth)])

# Threshold sweep - pick the best F1 (report the curve in the paper)
sweep = pd.DataFrame([{"threshold": th, **run_classical(th)[1]} for th in [0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5]])
best_th = float(sweep.loc[sweep.f1.idxmax(), "threshold"])
sweep.plot(x="threshold", y=["precision", "recall", "f1"], figsize=(7, 3.5), marker="o")
plt.axvline(best_th, ls="--", c="gray"); plt.title(f"Classical method - threshold sweep (best={best_th})")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "exp3_threshold_sweep.png", dpi=150); plt.show()
classical_preds, classical_agg = run_classical(best_th)
print("Classical:", {k: round(v, 3) for k, v in classical_agg.items()})

## 3. Method A - YOLO-based
Uses the latest trained weights from notebook 02. The detector's classes must match the product names
in `planogram.json` / `sessions.csv`.

In [ ]:
from ultralytics import YOLO

WEIGHTS = detection.latest_run_weights(which="best.pt")   # or set a specific run's best.pt
CONF = 0.35
yolo_preds, yolo_agg = None, None

if WEIGHTS is None:
    print("No trained weights yet - run notebook 02, then re-run this section.")
else:
    model = YOLO(str(WEIGHTS))
    missing = set(planogram.values()) - set(model.names.values())
    if missing:
        print("WARNING: these shelf products are not classes of the model:", missing)
    yolo_preds, scores = [], []
    for _, s in sessions.iterrows():
        b = cv2.imread(str(root / s.session_id / "before.jpg")); a = cv2.imread(str(root / s.session_id / "after.jpg"))
        r = cd.removed_yolo(model, b, a, conf=CONF, device=get_device())
        yolo_preds.append(r["removed"]); scores.append(cd.score_removals(r["removed"], s.truth))
    yolo_agg = cd.aggregate(scores)
    print("YOLO:", {k: round(v, 3) for k, v in yolo_agg.items()})

    show([draw_dets(r["before_img"], r["before_dets"]), draw_dets(r["after_img"], r["after_dets"])],
         [f"before - {len(r['before_dets'])} detections", f"after - removed: {r['removed']}"], size=5.5)

## 4. Experiment 3 - comparison + basket export

In [ ]:
rows = [{"method": "Classical (colour diff + planogram)", **classical_agg}]
if yolo_agg:
    rows.append({"method": "YOLOv8 (count difference)", **yolo_agg})
exp3 = pd.DataFrame(rows).round(3)
exp3.to_csv(RESULTS_DIR / "exp3_change_detection.csv", index=False)
display(exp3)

out = sessions[["session_id"]].copy()
out["truth"] = sessions.truth.apply(";".join)
out["classical"] = [";".join(p) for p in classical_preds]
if yolo_preds is not None:
    out["yolo"] = [";".join(p) for p in yolo_preds]
out["source"] = "real" if real else "demo"
out.to_csv(RESULTS_DIR / "detected_baskets.csv", index=False)
print("Saved baskets ->", RESULTS_DIR / "detected_baskets.csv")
out.head(10)

**For the paper (P2):** describe both methods, report the Experiment 3 table and the threshold curve,
and discuss failure cases (occlusion by hands, products moved not removed, product colour close to the shelf background).
The classical method needs a planogram; YOLO doesn't - that trade-off is a discussion point.